In [1]:
import sys
import os
import pandas as pd

src_path = os.path.abspath(os.path.join(os.path.dirname("__file__"), '..', 'src'))
if src_path not in sys.path:
    sys.path.append(src_path)

from dense_index_bge import DenseIndex
from sparse_index import SparseIndex


libgomp: Invalid value for environment variable OMP_NUM_THREADS


In [2]:
court_consideration_df = pd.read_csv("../data/court_considerations.csv")
court_consideration_d = dict(zip(court_consideration_df['citation'].tolist(), court_consideration_df['text'].tolist()))

law_df = pd.read_csv("../data/laws_de.csv")
law_d = dict(zip(law_df['citation'].tolist(), law_df['text'].tolist()))

court_doc = [{'citation':citation, 'text':text} for citation,text in zip(court_consideration_df['citation'].tolist(), court_consideration_df['text'].tolist())]
law_doc = [{'citation':citation, 'text':text} for citation,text in zip(law_df['citation'].tolist(), law_df['text'].tolist())]

test_df = pd.read_csv("../data/test_rewrite_001.csv")
_d = {}
for _, row in test_df.iterrows():
    if row['query_id'] not in _d:
        _d[row['query_id']] = [row['query']]
    else:
        _d[row['query_id']].append(row['query'])
test_dict = {k: v for k, v in sorted(_d.items())}

valid_df = pd.read_csv("../data/valid_rewrite_001.csv")

print("data loaded.")

data loaded.


In [3]:
from FlagEmbedding import FlagReranker, BGEM3FlagModel

model = BGEM3FlagModel('/root/.cache/modelscope/hub/models/BAAI/bge-m3', use_fp16=True, show_progress_bar=False)

dense_index = DenseIndex(model, "/root/autodl-fs/bge-processed/_dense_sparse_court/", court_doc)

sparse_index = SparseIndex(model, "/root/autodl-fs/bge-processed/_dense_sparse_court/", court_doc)

reranker = FlagReranker('/root/.cache/modelscope/hub/models/BAAI/bge-reranker-v2-m3', use_fp16=True, normalize=True)


libgomp: Invalid value for environment variable OMP_NUM_THREADS

libgomp: Invalid value for environment variable OMP_NUM_THREADS


DenseIndex.embeddings:  (2107648, 1024)


In [4]:
train_df = pd.read_csv("../data/train_rewrite_001.csv")

In [5]:
import citation_utils

In [6]:
import random

def gen_sample_for_query(query_id, query, gold_citation_l):
    gold_citation_set = set(gold_citation_l)
    cc_with_score_l = dense_index.search_with_score(query, 1000)
    cc_with_score_l = sorted(cc_with_score_l, key=lambda x: x[1], reverse=True)

    TOPK=50
    
    # 计算有gold的cc
    cc_contains_gold = set()
    for cc, score in cc_with_score_l[:TOPK]:
        for gold in gold_citation_set:
            if gold in cc['text']:
                cc_contains_gold.add(cc['citation'])

    # 算出所有的evidence，evidence会记录它属于哪个cc
    evidence_d = {}
    for cc, score in cc_with_score_l[:TOPK]:
        parsed_cc = citation_utils.parse_cc_output_citations_and_sentences(cc['text'])
        for citation, first_appear_idx in parsed_cc['citations']:
            evidence_text = citation_utils.build_evidence(parsed_cc['sentences'], first_appear_idx, 5)
            evidence_d[evidence_text] = {"cc_citation": cc['citation']}

    positive_evidence_set = set()
    for evidence_text, _ in evidence_d.items():
        for gold_citation in gold_citation_l:
            if gold_citation in evidence_text:
                positive_evidence_set.add(evidence_text)

    hard_negative_evidence_set = set()
    for evidence_text, meta in evidence_d.items():
        if evidence_text in positive_evidence_set:
            continue

        if meta["cc_citation"] in cc_contains_gold:
            continue
            
        hard_negative_evidence_set.add(evidence_text)
        
    pos_cnt = len(positive_evidence_set)
    # 4 个hard negative
    hard_negative_evidence_list = list(hard_negative_evidence_set)
    random.shuffle(hard_negative_evidence_list)

    pos_l = list(positive_evidence_set)
    neg_l = hard_negative_evidence_list[:pos_cnt*4]

    rand_l = []
    # 2 个random negative
    l1 = cc_with_score_l[:500].copy()
    random.shuffle(l1)
    for cc, score in l1:
        parsed_cc = citation_utils.parse_cc_output_citations_and_sentences(cc['text'])
        if len(parsed_cc['citations']) > 0:
            citation, first_appear_idx = parsed_cc['citations'][0]
            evidence_text = citation_utils.build_evidence(parsed_cc['sentences'], first_appear_idx, 5)
            rand_l.append(evidence_text)
        if len(rand_l) >= pos_cnt*2:
            break
            

    return pos_l, neg_l, rand_l
    
    # print(query_id, 'positive_evidence_set.len:', len(positive_evidence_set), 
    #       "hard_negative_evidence_set.len:", len(hard_negative_evidence_set),
    #       "gold.len:", len(gold_citation_set))

In [7]:
import random

def gen_sample_for_query2(query_id, query, gold_citation_l):
    gold_citation_set = set(gold_citation_l)
    cc_with_score_l = dense_index.search_with_score(query, 1000)
    cc_with_score_l = sorted(cc_with_score_l, key=lambda x: x[1], reverse=True)

    TOPK=10
    
    # 计算有gold的cc
    cc_contains_gold = set()
    for cc, score in cc_with_score_l[:TOPK]:
        for gold in gold_citation_set:
            if gold in cc['text']:
                cc_contains_gold.add(cc['citation'])

    # 算出TOPK的evidence，evidence会记录它属于哪个cc
    evidence_d = {}
    for cc, score in cc_with_score_l[:TOPK]:
        parsed_cc = citation_utils.parse_cc_output_citations_and_sentences(cc['text'])
        for citation, first_appear_idx in parsed_cc['citations']:
            evidence_text = citation_utils.build_evidence(parsed_cc['sentences'], first_appear_idx, 5)
            evidence_d[evidence_text] = {"cc_citation": cc['citation']}

    positive_evidence_set = set()
    for evidence_text, _ in evidence_d.items():
        for gold_citation in gold_citation_l:
            if gold_citation in evidence_text:
                positive_evidence_set.add(evidence_text)

    # TOPK to TOPK + 30，排除掉有gold的cc
    evidence_d = {}
    for cc, score in cc_with_score_l[TOPK:TOPK+30]:
        has_gold = False
        for gold in gold_citation_set:
            if gold in cc['text']:
                has_gold = True
                break
        if has_gold:
            continue #排除掉有gold的cc
        else:
            parsed_cc = citation_utils.parse_cc_output_citations_and_sentences(cc['text'])
            for citation, first_appear_idx in parsed_cc['citations']:
                evidence_text = citation_utils.build_evidence(parsed_cc['sentences'], first_appear_idx, 5)
                evidence_d[evidence_text] = {"cc_citation": cc['citation']}

    hard_negative_evidence_set = set()
    for evidence_text, meta in evidence_d.items():
        if evidence_text in positive_evidence_set:
            continue

        if meta["cc_citation"] in cc_contains_gold:
            continue
            
        hard_negative_evidence_set.add(evidence_text)
        
    pos_cnt = len(positive_evidence_set)
    # 4 个hard negative
    hard_negative_evidence_list = list(hard_negative_evidence_set)
    random.shuffle(hard_negative_evidence_list)

    pos_l = list(positive_evidence_set)
    neg_l = hard_negative_evidence_list[:pos_cnt*3]

    rand_l = []
    # 2 个random negative
    l1 = cc_with_score_l[:500].copy()
    random.shuffle(l1)
    for cc, score in l1:
        parsed_cc = citation_utils.parse_cc_output_citations_and_sentences(cc['text'])
        if len(parsed_cc['citations']) > 0:
            citation, first_appear_idx = parsed_cc['citations'][0]
            evidence_text = citation_utils.build_evidence(parsed_cc['sentences'], first_appear_idx, 5)
            rand_l.append(evidence_text)
        if len(rand_l) >= pos_cnt*6:
            break
            

    return pos_l, neg_l, rand_l
    
    # print(query_id, 'positive_evidence_set.len:', len(positive_evidence_set), 
    #       "hard_negative_evidence_set.len:", len(hard_negative_evidence_set),
    #       "gold.len:", len(gold_citation_set))

In [8]:
import json
all_sample = []
for query_id, query, gold_citations in zip(train_df['query_id'], train_df['query2'], train_df['gold_citations']):
    pos_l, neg_l, rand_l = gen_sample_for_query2(query_id, query, gold_citations.split(';'))
    if len(pos_l) > 0:
        print(query_id, ", pos_l.len:", len(pos_l), ", neg_l.len:", len(neg_l), ", rand_l.len:", len(rand_l))
    for evidence in pos_l:
        all_sample.append({"query":query, "passage": evidence, "label":"1"})
    for evidence in neg_l:
        all_sample.append({"query":query, "passage": evidence, "label":"0"})
    # for evidence in rand_l:
    #    all_sample.append({"query":query, "passage": evidence, "label":"0"})

random.shuffle(all_sample)

train_size = int(len(all_sample)*0.9)

train = all_sample[:train_size]
valid = all_sample[train_size:]

with open("../ft_data/train.jsonl", "w+", encoding="utf-8") as of:
    for sample in train:
        of.write(json.dumps(sample, ensure_ascii=False)+"\n")

with open("../ft_data/valid.jsonl", "w+", encoding="utf-8") as of:
    for sample in valid:
        of.write(json.dumps(sample, ensure_ascii=False)+"\n")
      
    

You're using a XLMRobertaTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


train_0002 , pos_l.len: 9 , neg_l.len: 27 , rand_l.len: 54
train_0004 , pos_l.len: 4 , neg_l.len: 12 , rand_l.len: 24
train_0005 , pos_l.len: 12 , neg_l.len: 36 , rand_l.len: 72
train_0006 , pos_l.len: 15 , neg_l.len: 45 , rand_l.len: 90
train_0007 , pos_l.len: 23 , neg_l.len: 62 , rand_l.len: 138
train_0009 , pos_l.len: 12 , neg_l.len: 16 , rand_l.len: 72
train_0010 , pos_l.len: 5 , neg_l.len: 15 , rand_l.len: 30
train_0014 , pos_l.len: 18 , neg_l.len: 44 , rand_l.len: 108
train_0016 , pos_l.len: 16 , neg_l.len: 48 , rand_l.len: 96
train_0018 , pos_l.len: 12 , neg_l.len: 36 , rand_l.len: 72
train_0022 , pos_l.len: 1 , neg_l.len: 3 , rand_l.len: 6
train_0023 , pos_l.len: 3 , neg_l.len: 9 , rand_l.len: 18
train_0025 , pos_l.len: 1 , neg_l.len: 3 , rand_l.len: 6
train_0026 , pos_l.len: 9 , neg_l.len: 27 , rand_l.len: 54
train_0028 , pos_l.len: 3 , neg_l.len: 9 , rand_l.len: 18
train_0030 , pos_l.len: 12 , neg_l.len: 36 , rand_l.len: 72
train_0031 , pos_l.len: 2 , neg_l.len: 6 , rand_l.le